# SQL Environment & Database Integration Workflow

This notebook demonstrates connecting Python data pipelines to SQL databases:
1. **Setting up SQLAlchemy Database Connections** (SQLite file-based & PostgreSQL connection strings).
2. **Loading Cleaned DataFrames to SQL Tables** (`to_sql` with `if_exists='replace'`).
3. **Inspecting and Validating Database Schemas** (`sqlalchemy.inspect`).
4. **Querying Database Tables with Pandas** (`pd.read_sql` for filtered & aggregate queries).
5. **Building Repeatable Modular Database Loading Functions**.

In [1]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, inspect, text

db_path = '../analytics.db'
if not os.path.exists(db_path):
    db_path = 'analytics.db'

engine = create_engine(f'sqlite:///{db_path}')
with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
    print("[SUCCESS] Connected to database engine successfully!")

## Task 1 & Task 2: Loading Cleaned Data to SQL Table

In [2]:
np.random.seed(42)
num_records = 1000
df_clean = pd.DataFrame({
    'customer_id': range(1001, 1001 + num_records),
    'customer_name': [f"Customer_{i}" for i in range(1001, 1001 + num_records)],
    'customer_type': np.random.choice(['Enterprise', 'SMB', 'Startup'], size=num_records, p=[0.15, 0.45, 0.40]),
    'email': [f"user_{i}@company.com" for i in range(1001, 1001 + num_records)],
    'signup_date': pd.date_range(start='2024-01-01', periods=num_records, freq='8h').strftime('%Y-%m-%d'),
    'lifetime_value': np.round(np.random.exponential(scale=15000, size=num_records) + 1000, 2),
    'churn': np.random.choice([0, 1], size=num_records, p=[0.92, 0.08])
})

df_clean.to_sql('customers_cleaned', engine, if_exists='replace', index=False)

inspector = inspect(engine)
print(f"Database Tables: {inspector.get_table_names()}")
row_count = pd.read_sql("SELECT COUNT(*) as ct FROM customers_cleaned", engine).iloc[0]['ct']
print(f"Loaded {row_count:,} rows into 'customers_cleaned'.")

## Task 3: Schema Inspection & Validation

In [3]:
columns = inspector.get_columns('customers_cleaned')
print("TABLE COLUMNS & DATA TYPES:")
for col in columns:
    nullable = "NOT NULL" if not col.get('nullable', True) else "NULLABLE"
    print(f"  - {col['name']:20} : {str(col['type']):15} ({nullable})")

## Task 4: Analytical Query Execution

In [4]:
query_enterprise = "SELECT * FROM customers_cleaned WHERE customer_type = 'Enterprise'"
enterprise_df = pd.read_sql(query_enterprise, engine)
print(f"Retrieved {len(enterprise_df)} Enterprise customers.")

query_agg = """
SELECT 
    customer_type, 
    COUNT(*) as count, 
    ROUND(AVG(lifetime_value), 2) as avg_ltv
FROM customers_cleaned 
GROUP BY customer_type 
ORDER BY avg_ltv DESC
"""
summary_df = pd.read_sql(query_agg, engine)
print("\nSegment Aggregation Summary:")
print(summary_df)

## Task 5: Modular Repeatable Database Loader Function

In [5]:
def load_cleaned_data_to_database(df, table_name, database_path='analytics.db'):
    """Modular function to write DataFrame to SQL database and validate row count."""
    db_engine = create_engine(f'sqlite:///{database_path}')
    df.to_sql(table_name, db_engine, if_exists='replace', index=False)
    loaded = pd.read_sql(f"SELECT COUNT(*) as ct FROM {table_name}", db_engine).iloc[0]['ct']
    print(f"[SUCCESS] Loaded {loaded:,} rows to '{table_name}' table in '{database_path}'.")
    return db_engine

engine_test = load_cleaned_data_to_database(df_clean, 'customers_cleaned', db_path)
test_df = pd.read_sql("SELECT customer_type, COUNT(*) as count FROM customers_cleaned GROUP BY customer_type", engine_test)
print(test_df)